# Cap Sweep — Random Token Rejection

**Goal.** Find the optimal cap ρ\* for Random Token Rejection (RTR) before running the §3 main table.

**Configs.** Random selection × cap ρ ∈ {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7} × step counts {8, 16} × seeds {0, 1, 2} = **42 FID-10K configs**.

**Output.** A single recommended ρ\* (and the cap-saturation curves) that will populate `RTR_CAP` in `main_table_colab.ipynb`.

**Persistence.** Master CSV at `MyDrive/ARPG-assets/results/final-paper/cap-sweep-random/results.csv`. Same resumability as the main-table notebook: re-run end-to-end any time, completed configs are skipped.

**Estimated time.** ~5–6 hours of A100 time (fits in a single Colab Pro overnight session).

**Disk.** Each FID-10K NPZ is ~2 GB. 42 NPZs ≈ 84 GB of new Drive data.

## 1. Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')

# Output location for this notebook
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'cap-sweep-random'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_ROOT / 'results.csv'
NPZ_DIR = RESULTS_ROOT / 'random-10k'
NPZ_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = RESULTS_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Rejection tracker output (small JSON + heatmap PNG per config; kept on Drive)
REJECTION_LOGS_DIR = RESULTS_ROOT / 'rejection-logs'
REJECTION_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# NPZ persistence policy.
# False = keep only FID metrics + rejection JSON; delete the 2 GB FID-10K NPZ after eval.
# True  = also archive every NPZ to Drive (84 GB total for this sweep).
KEEP_NPZ_ON_DRIVE = False

REPO_LOCAL = Path('/content/ARPG-main')
LOCAL_SAMPLE_DIR = Path('/content/samples')
LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

REF_NPZ = DRIVE_ROOT / 'eval' / 'VIRTUAL_imagenet256_labeled.npz'
ARPG_CKPT = DRIVE_ROOT / 'weights' / 'arpg_300m.pt'
VQ_CKPT = DRIVE_ROOT / 'weights' / 'vq_ds16_c2i.pt'
GD_REPO = DRIVE_ROOT / 'external' / 'guided-diffusion'

print(f'Results root: {RESULTS_ROOT}')
print(f'Master CSV  : {CSV_PATH}')
print(f'NPZ folder  : {NPZ_DIR}')

## 2. Clone repo and verify assets (auto-downloads anything missing)

In [ ]:
# Private fork containing the random-deferral support
REPO_URL = 'https://github.com/rshahbazov23/comp447-arpg-private.git'

# Set a GitHub PAT if the repo is still private.
GITHUB_TOKEN = None  # e.g. 'ghp_...'

import subprocess, shutil

def _clone_url(url, token):
    if token and url.startswith('https://github.com/'):
        return url.replace('https://', f'https://{token}@')
    return url

# --- 1. Clone / update the ARPG fork ---------------------------------------
if not REPO_LOCAL.exists():
    print(f'Cloning {REPO_URL} → {REPO_LOCAL}')
    subprocess.run(['git', 'clone', _clone_url(REPO_URL, GITHUB_TOKEN), str(REPO_LOCAL)], check=True)
else:
    print(f'Repo already present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_LOCAL), 'pull'], check=True)

# --- 2. Auto-setup assets on Drive -----------------------------------------
ASSET_URLS = {
    REF_NPZ:   'https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz',
    ARPG_CKPT: 'https://huggingface.co/hp-l33/ARPG/resolve/main/arpg_300m.pt',
    VQ_CKPT:   'https://huggingface.co/FoundationVision/LlamaGen/resolve/main/vq_ds16_c2i.pt',
}

ALT_LOCATIONS = [
    Path('/content/drive/MyDrive/eval'),
    Path('/content/drive/MyDrive/weights'),
    Path('/content/drive/MyDrive/ARPG/eval'),
    Path('/content/drive/MyDrive/ARPG/weights'),
    Path('/content/drive/MyDrive/ARPG-main/eval'),
    Path('/content/drive/MyDrive/ARPG-main/weights'),
]

def find_alt(filename):
    for root in ALT_LOCATIONS:
        cand = root / filename
        if cand.exists():
            return cand
    return None

for target, url in ASSET_URLS.items():
    if target.exists():
        size_gb = target.stat().st_size / 1e9
        print(f'OK   {target.name:<40} ({size_gb:.2f} GB)')
        continue
    alt = find_alt(target.name)
    if alt:
        print(f'Found {target.name} at {alt} — copying to {target}')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(alt, target)
        continue
    print(f'Downloading {target.name} → {target}')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '--show-progress', '-O', str(target), url], check=True)
    print(f'   downloaded ({target.stat().st_size/1e9:.2f} GB)')

# --- 3. guided-diffusion ---------------------------------------------------
if not GD_REPO.exists():
    alt_gd = None
    for root in [Path('/content/drive/MyDrive/external/guided-diffusion'),
                 Path('/content/drive/MyDrive/guided-diffusion'),
                 Path('/content/drive/MyDrive/ARPG/external/guided-diffusion')]:
        if root.exists() and (root / 'evaluations' / 'evaluator.py').exists():
            alt_gd = root
            break
    if alt_gd:
        print(f'Found guided-diffusion at {alt_gd} — copying to {GD_REPO}')
        GD_REPO.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(alt_gd, GD_REPO)
    else:
        print(f'Cloning guided-diffusion → {GD_REPO}')
        GD_REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', 'https://github.com/openai/guided-diffusion.git', str(GD_REPO)], check=True)

# --- 4. Sanity check -------------------------------------------------------
required = [
    (REF_NPZ,   'ImageNet reference batch'),
    (ARPG_CKPT, 'ARPG-L pretrained checkpoint'),
    (VQ_CKPT,   'LlamaGen VQ tokenizer'),
    (GD_REPO / 'evaluations' / 'evaluator.py', 'guided-diffusion evaluator'),
    (REPO_LOCAL / 'sample_c2i_ddp.py', 'ARPG sampling script'),
    (REPO_LOCAL / 'models' / 'arpg.py', 'ARPG model'),
    (REPO_LOCAL / 'models' / 'confidence.py', 'Confidence metrics module'),
]
for p, name in required:
    if not p.exists():
        raise FileNotFoundError(f'MISSING: {name} — expected at {p}')
print('\nAll assets present.')

conf_src = (REPO_LOCAL / 'models' / 'confidence.py').read_text()
if 'random_score' not in conf_src:
    raise RuntimeError('confidence.py does not include random selection. '
                       'Check out commit 605038a or later.')
print('Random-deferral support confirmed.')

## 3. Install Python dependencies

In [ ]:
subprocess.run(['pip', 'install', '-q',
    'einops', 'transformers', 'scipy', 'tensorflow', 'pandas',
], check=True)

import torch, einops, transformers, scipy, pandas as pd
print(f'torch        : {torch.__version__}, CUDA {torch.version.cuda}, GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'transformers : {transformers.__version__}')
print(f'pandas       : {pd.__version__}')
assert torch.cuda.is_available(), 'No CUDA device — switch the Colab runtime to GPU.'

## 4. Config matrix and master CSV

42 configs total: random selection × 7 caps × 2 step counts × 3 seeds.

Configs run in this order (priority): 16 steps first (the headline regime where ρ\* will be selected), then 8 steps (cross-check that ρ\* generalises).

In [ ]:
import pandas as pd
from datetime import datetime

STEP_COUNTS_ORDER = [16, 8]
CAPS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
SEEDS = [0, 1, 2]

RTR_METRIC = 'random'
RTR_THRESHOLD = 2.0  # unreachable — forces the cap branch
NUM_FID_SAMPLES = 10_000

def make_configs():
    out = []
    for step in STEP_COUNTS_ORDER:
        for cap in CAPS:
            for seed in SEEDS:
                out.append({'step': step, 'cap': cap, 'seed': seed})
    return out

CONFIGS = make_configs()
print(f'Config matrix: {len(CONFIGS)} configs (2 steps × 7 caps × 3 seeds)')

# Initialise master CSV (empty if first run; otherwise preserve existing rows)
if not CSV_PATH.exists():
    pd.DataFrame(columns=[
        'step', 'cap', 'seed', 'fid',
        'inception_score', 'sfid', 'precision', 'recall',
        'npz_path', 'timestamp',
    ]).to_csv(CSV_PATH, index=False)
    print(f'Initialised empty master CSV: {CSV_PATH}')
else:
    df_master = pd.read_csv(CSV_PATH)
    print(f'Master CSV has {len(df_master)} existing rows.')

df_master = pd.read_csv(CSV_PATH)
done_keys = set(
    (int(r['step']), float(r['cap']), int(r['seed']))
    for _, r in df_master.iterrows()
) if len(df_master) else set()
remaining = [c for c in CONFIGS
             if (c['step'], c['cap'], c['seed']) not in done_keys]
print(f'\nProgress: {len(CONFIGS) - len(remaining)}/{len(CONFIGS)} done, {len(remaining)} remaining\n')
if remaining[:5]:
    print('First 5 configs to run:')
    for c in remaining[:5]:
        print(f'  {c}')

## 5. Helper functions

In [ ]:
import re, time, traceback

def config_to_folder_name(cfg):
    """Mirrors sample_c2i_ddp.py:114 naming convention."""
    base = (
        f'ARPG-L-arpg_300m-size-256-size-256-VQ-16-'
        f'topk-0-topp-1.0-temperature-1.0-cfg-5.0-cfg-schedule-linear-'
        f'sample-schedule-arccos-step-{cfg["step"]}-seed-{cfg["seed"]}-'
        f'mode-rejection-metric-{RTR_METRIC}-tau-{RTR_THRESHOLD}-cap-{cfg["cap"]}'
    )
    return base


def is_done(cfg, df_master):
    if not len(df_master):
        return False
    mask = (
        (df_master['step'].astype(int) == cfg['step'])
        & (df_master['cap'].astype(float).round(2) == round(cfg['cap'], 2))
        & (df_master['seed'].astype(int) == cfg['seed'])
    )
    return bool(mask.any())


def cleanup_local_samples():
    if LOCAL_SAMPLE_DIR.exists():
        shutil.rmtree(LOCAL_SAMPLE_DIR)
    LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)


def run_sampling(cfg, log_handle=None):
    folder = config_to_folder_name(cfg)
    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', 'ARPG-L',
        '--gpt-ckpt', str(ARPG_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', '5.0',
        '--step', str(cfg['step']),
        '--per-proc-batch-size', '64',
        '--num-fid-samples', str(NUM_FID_SAMPLES),
        '--global-seed', str(cfg['seed']),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
        '--rejection-mode', 'rejection',
        '--confidence-metric', RTR_METRIC,
        '--rejection-threshold', str(RTR_THRESHOLD),
        '--max-reject-rate', str(cfg['cap']),
        '--log-json', str(REJECTION_LOGS_DIR / f'{folder}.json'),
    ]
    print(f'  Sampling: step={cfg["step"]} cap={cfg["cap"]} seed={cfg["seed"]}')
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_LOCAL),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    last_print = t0
    for line in proc.stdout:
        if log_handle is not None:
            log_handle.write(line); log_handle.flush()
        now = time.time()
        if now - last_print > 60:
            print(f'    [{(now-t0)/60:.1f} min] {line.rstrip()[:120]}')
            last_print = now
    proc.wait()
    print(f'  Sampling done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if proc.returncode != 0:
        raise RuntimeError(f'Sampling failed for {cfg}: exit {proc.returncode}')
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    if not local_npz.exists():
        raise FileNotFoundError(f'Expected NPZ not produced: {local_npz}')
    return local_npz


def sync_npz_to_drive(local_npz):
    if not KEEP_NPZ_ON_DRIVE:
        return None
    drive_npz = NPZ_DIR / local_npz.name
    size_gb = local_npz.stat().st_size / 1e9
    print(f'  Syncing {local_npz.name} ({size_gb:.2f} GB) → Drive…')
    t0 = time.time()
    shutil.copy2(local_npz, drive_npz)
    print(f'  Drive sync done in {time.time()-t0:.1f} s')
    return drive_npz


_METRIC_LINE = re.compile(r'^\s*(FID|sFID|Inception Score|Precision|Recall)\s*:\s*([0-9.eE+\-]+)')

def evaluate_fid(local_npz, log_handle=None):
    cmd = ['python', 'evaluations/evaluator.py', str(REF_NPZ), str(local_npz)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(GD_REPO), capture_output=True, text=True)
    print(f'  FID eval done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if log_handle is not None:
        log_handle.write('--- evaluator stdout ---\n')
        log_handle.write(proc.stdout)
        log_handle.write('\n--- evaluator stderr ---\n')
        log_handle.write(proc.stderr)
        log_handle.flush()
    if proc.returncode != 0:
        print('STDOUT (tail):', proc.stdout[-2000:])
        print('STDERR (tail):', proc.stderr[-2000:])
        raise RuntimeError(f'FID eval failed: exit {proc.returncode}')
    metrics = {}
    for line in proc.stdout.splitlines():
        m = _METRIC_LINE.match(line)
        if m:
            key = m.group(1).lower().replace(' ', '_')
            metrics[key] = float(m.group(2))
    if 'fid' not in metrics:
        print('STDOUT:', proc.stdout)
        raise ValueError('Could not parse FID from evaluator output')
    return metrics


def append_result(cfg, metrics, npz_path):
    df = pd.read_csv(CSV_PATH)
    row = {
        'step': cfg['step'],
        'cap': cfg['cap'],
        'seed': cfg['seed'],
        'fid': metrics.get('fid'),
        'inception_score': metrics.get('inception_score'),
        'sfid': metrics.get('sfid'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'npz_path': str(npz_path) if npz_path is not None else '(not kept)',
        'timestamp': datetime.now().isoformat(),
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return df


print('Helpers loaded.')

## 6. Main loop — resumable

Safe to interrupt at any time. Each completed config writes a row to the Drive CSV immediately. Re-running this cell picks up where it left off.

In [ ]:
failures = []
started = datetime.now()

for i, cfg in enumerate(CONFIGS, 1):
    print(f'\n{"="*70}\n[{i}/{len(CONFIGS)}] config={cfg}\n{"="*70}')

    df_master = pd.read_csv(CSV_PATH)
    if is_done(cfg, df_master):
        print('  SKIP (already in CSV)')
        continue

    folder = config_to_folder_name(cfg)
    drive_npz_existing = NPZ_DIR / f'{folder}.npz'
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    log_path = LOG_DIR / f'{folder}.log'

    try:
        with open(log_path, 'w') as log_h:
            if drive_npz_existing.exists():
                print(f'  NPZ already on Drive ({drive_npz_existing.stat().st_size/1e9:.2f} GB) — re-using for eval')
                shutil.copy2(drive_npz_existing, local_npz)
                drive_npz = drive_npz_existing
            else:
                local_npz = run_sampling(cfg, log_handle=log_h)
                drive_npz = sync_npz_to_drive(local_npz)  # None if KEEP_NPZ_ON_DRIVE=False

            metrics = evaluate_fid(local_npz, log_handle=log_h)
            append_result(cfg, metrics, drive_npz)
        print(f'  DONE   FID={metrics["fid"]:.4f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        failures.append({'config': cfg, 'error': str(e), 'log': str(log_path)})
    finally:
        cleanup_local_samples()

    elapsed = (datetime.now() - started).total_seconds() / 3600
    print(f'  Session elapsed: {elapsed:.2f} h')

print(f'\n\n{"="*70}\nSESSION COMPLETE\n{"="*70}')
print(f'Configs attempted: {len(CONFIGS)}')
print(f'Failures: {len(failures)}')
for f in failures:
    print(f'  {f["config"]}  →  {f["error"]}  (log: {f["log"]})')

## 7. Pick ρ\* — cap-saturation curves and recommended cap for §3

Reads the master CSV, computes mean FID across seeds at each (step, cap), prints the curves, and outputs a single recommended ρ\* for the main-table notebook.

In [ ]:
import numpy as np

df = pd.read_csv(CSV_PATH)
if len(df) == 0:
    raise RuntimeError('Master CSV is empty — run the main loop first.')

df['step'] = df['step'].astype(int)
df['cap'] = df['cap'].astype(float).round(2)
df['seed'] = df['seed'].astype(int)

print(f'Master CSV: {len(df)} rows\n')

# ---- Per-step cap curves --------------------------------------------------
summary = (
    df.groupby(['step', 'cap'])['fid']
      .agg(['mean', 'std', 'count'])
      .round(4)
      .sort_index()
)
print('FID-10K — cap-saturation curves (mean ± std, n)\n')
print(summary.to_string())

# ---- Pick ρ* per step -----------------------------------------------------
best_per_step = {}
for step in sorted(df['step'].unique()):
    sub = df[df['step'] == step].groupby('cap')['fid'].mean()
    if len(sub) == 0:
        continue
    best_cap = float(sub.idxmin())
    best_fid = float(sub.min())
    best_per_step[int(step)] = (best_cap, best_fid)

print('\n\nBest ρ per step count:')
for step, (cap, fid) in sorted(best_per_step.items()):
    print(f'  {step:>3} steps:   ρ* = {cap:.2f}   (mean FID-10K = {fid:.4f})')

# ---- Recommend a single ρ* for the main table -----------------------------
# Prefer the 16-step optimum since that’s the headline regime; verify the 8-step
# optimum is within FID-10K noise of the 16-step optimum.
if 16 not in best_per_step:
    raise RuntimeError('No 16-step data yet — finish the 16-step block first.')

rho_star = best_per_step[16][0]
rho_star_fid_16 = best_per_step[16][1]

print(f'\n\n{"="*70}')
print(f'RECOMMENDATION for §3 main table:   RTR_CAP = {rho_star:.2f}')
print(f'{"="*70}')
print(f'  16-step ρ* = {rho_star:.2f}   (FID-10K {rho_star_fid_16:.4f})')
if 8 in best_per_step:
    rho_star_8, fid_at_8 = best_per_step[8]
    fid_at_8_with_16_rho = (
        df[(df['step'] == 8) & (df['cap'] == rho_star)]['fid'].mean()
    )
    print(f'  8-step  ρ* = {rho_star_8:.2f}   (FID-10K {fid_at_8:.4f})')
    if not np.isnan(fid_at_8_with_16_rho):
        print(f'  8-step  FID at the 16-step ρ* ({rho_star:.2f}) = {fid_at_8_with_16_rho:.4f}')
        gap = fid_at_8_with_16_rho - fid_at_8
        if gap < 0.05:
            print(f'  → 8-step optimum is within {gap:.4f} FID of the 16-step ρ* — use ρ* = {rho_star:.2f} for all step counts.')
        else:
            print(f'  → 8-step optimum differs by {gap:.4f} FID — consider step-specific ρ in the main table.')

# ---- Save summary CSV alongside master -----------------------------------
summary_path = RESULTS_ROOT / 'summary.csv'
summary.to_csv(summary_path)
print(f'\nSummary written to {summary_path}')

# ---- Quick visual: cap-saturation curve ----------------------------------
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for step in sorted(df['step'].unique()):
        agg = df[df['step'] == step].groupby('cap')['fid'].agg(['mean', 'std']).reset_index()
        ax.errorbar(agg['cap'], agg['mean'], yerr=agg['std'],
                    label=f'{step} steps', marker='o', capsize=3)
    ax.set_xlabel(r'cap $\rho$')
    ax.set_ylabel('FID-10K (mean ± std, n=3)')
    ax.set_title('RTR cap-saturation curves — random selection')
    ax.legend()
    ax.grid(alpha=0.3)
    fig_path = RESULTS_ROOT / 'cap_saturation.png'
    fig.tight_layout()
    fig.savefig(fig_path, dpi=120)
    print(f'Plot saved to {fig_path}')
    plt.show()
except Exception as e:
    print(f'(Plotting skipped: {e})')